# Piano B — Notebook H: label da self-report, feature e TSception (Sessione 03)

1. Passo 0: tabella dei voti dai file After_remarks (allineamento delle righe deciso dai dati, controlli
   bloccanti, hash). Se un controllo fallisce il notebook si ferma con il motivo: riportalo.
2. H1: statistiche descrittive dei voti -> `report_selfreport_stage1.md` (scritto subito in /kaggle/working).
3. H2: rianalisi delle predizioni a 9 classi già salvate dalle sessioni precedenti (zero training).
4. Coda `H`: DE + z-score (worker solo-CPU, con le permutazioni entro clip) e TSception (GPU).
5. Analisi finale: tabelle 2 x 2, terza famiglia, `results/selfreport/`.

Impostazioni: Accelerator **GPU T4 x2**, Internet off. Input: `pianob-faced-data` (con i file comportamentali),
`pianob-code`, gli **output di tutte le sessioni precedenti** (servono alla rianalisi H2; restano in sola
lettura) e, per riprendere, l'output della sessione precedente di questo notebook.

In [ ]:
import os
os.environ["PIANOB_ACCOUNT"] = "nbH_selfreport_feature"   # la ripresa usa solo gli output di questo notebook

In [ ]:
# Setup: trova il codice (Dataset "pianob-code"), lo copia fuori da /kaggle/working (così non finisce
# negli output) e lo aggiunge a sys.path. Non serve modificare nulla qui.
import glob, os, shutil, sys, time
os.environ.setdefault("PIANOB_T0", str(time.time()))      # inizio della sessione (guardia delle 12 h)
os.environ.setdefault("PIANOB_BUDGET_H", "11.3")           # stop pulito prima del limite Kaggle
SYNTHETIC = os.environ.get("PIANOB_SYNTHETIC") == "1"      # solo per prove locali


def _find_code():
    cands = []
    roots = ["/kaggle/input"] if os.path.exists("/kaggle/input") else []
    roots += [os.getcwd(), os.path.dirname(os.getcwd())]
    for root in roots:
        for p in glob.glob(os.path.join(root, "**", "pianob", "__init__.py"), recursive=True):
            base = os.path.dirname(os.path.dirname(p))
            if "pianob_state" not in base:
                cands.append(base)
    assert cands, "Codice non trovato: aggiungi il Dataset 'pianob-code' come input (Add Input)."
    # se ci sono più copie, usa quella con VERSION più recente
    def _ver(b):
        v = os.path.join(b, "VERSION")
        return open(v).read().strip() if os.path.exists(v) else ""
    return sorted(cands, key=_ver)[-1]


CODE_SRC = _find_code()
if os.path.exists("/kaggle/working"):
    CODE = "/tmp/pianob_code"
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(CODE_SRC, CODE, ignore=shutil.ignore_patterns("notebooks", "*.zip", "local_work"))
else:
    CODE = CODE_SRC
sys.path.insert(0, CODE)
os.chdir(CODE)
print("codice da", CODE_SRC, "| versione", open(os.path.join(CODE, "VERSION")).read().strip()
      if os.path.exists(os.path.join(CODE, "VERSION")) else "(locale)")

from pianob import kaggle_env, nb_helpers  # noqa: E402
nb_helpers.print_env()
kaggle_env.restore_state_from_inputs()

In [ ]:
# Passo 0 + H1: tabella dei voti e descrittive (CPU, pochi minuti)
from pathlib import Path
from pianob.selfreport import analysis as SA, table as ST
ACCOUNT = os.environ["PIANOB_ACCOUNT"]
META = ST.run_step0(synthetic=SYNTHETIC)
VOTES = ST.load_votes()
DESC = SA.descriptives(VOTES, META, n_boot=200 if SYNTHETIC else 1000)
OUT = kaggle_env.state_dir() / "results" / "selfreport"
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "descriptives.json").write_text(__import__("json").dumps(DESC, indent=1, default=float))
STAGE1 = SA.stage1_markdown(DESC)
(kaggle_env.working_dir() / "report_selfreport_stage1.md").write_text(STAGE1)
print(STAGE1)

In [ ]:
# H2: rianalisi delle predizioni a 9 classi delle sessioni precedenti (input in sola lettura + stato)
SOURCES = [p for p in kaggle_env.input_roots()] + [kaggle_env.state_dir()]
if not (OUT / "reanalysis_h2.json").exists():
    H2 = SA.reanalyze_emo9(SOURCES, VOTES, n_boot=200 if SYNTHETIC else 2000, n_perm=200 if SYNTHETIC else 5000)
    (OUT / "reanalysis_h2.json").write_text(__import__("json").dumps(H2, indent=1, default=float))
    print("\n".join(SA.session03_markdown([], H2)))

In [ ]:
nb_helpers.run_queue("H", gpus="auto", synthetic=SYNTHETIC, cpu_workers=2)
nb_helpers.queue_status("H")

In [ ]:
# Analisi finale (predizioni di questo notebook): tabelle 2 x 2, terza famiglia
from pianob import report
RES = SA.run_all([kaggle_env.state_dir()], OUT, rows=report.load_rows(), h2=False,
                 n_boot=200 if SYNTHETIC else 2000, n_perm=200 if SYNTHETIC else 5000)
print("\n".join(RES.get("markdown", [])))

In [ ]:
nb_helpers.finish(ACCOUNT)